# Genius 55M: инференс модели, обученной с нуля

Откройте ноутбук из корня репозитория или папки `notebooks/`. Используйте окружение из README и предварительно скачайте модель из релиза. Поддерживаются Linux и Windows; для инференса подходят CUDA и CPU.

Это продолжение текста после претрейна, без SFT и чат-шаблона.

In [1]:
import sys
from pathlib import Path

project = next(
    path for path in [Path.cwd(), *Path.cwd().parents]
    if (path / "src/inference.py").is_file() and (path / "configs/pretrain.json").is_file()
)
sys.path.insert(0, str(project / "src"))
from common import ROOT, read_json, write_json, file_hash
from inference import load_model, generate

checkpoint = ROOT / "checkpoints/genius55m/best"
examples = read_json(ROOT / "reports/paraphrases/successful_paraphrases.json")
assert file_hash(checkpoint / "model.safetensors") == examples["weights_sha256"]
model, tokenizer = load_model(checkpoint)
print("Чекпоинт: artifacts/checkpoints/genius55m/best")
print("Параметры:", sum(p.numel() for p in model.parameters()))
print("Устройство:", model.device)
print("Контекст:", model.config.max_position_embeddings, "токенов")


Чекпоинт: artifacts/checkpoints/genius55m/best
Параметры: 54817920
Устройство: cuda:0
Контекст: 1024 токенов


## Проверенный пример

Префикс «За зимой приходит» взят из `artifacts/reports/paraphrases/successful_paraphrases.json`. Сравниваем первое короткое дополнение с сохранённым результатом. Ниже также выводится полный ответ: последующие строки могут повторяться.

In [2]:
prompt = "За зимой приходит"
reference = next(
    item
    for group in examples["examples"]
    for item in group["successful_formulations"]
    if item["prompt"] == prompt
)
result = generate(model, tokenizer, prompt, seed=42, max_new_tokens=32, temperature=0)
first_line = next(line.strip() for line in result["continuation"].splitlines() if line.strip())
print("Префикс:", prompt)
print("Короткое дополнение:", first_line)
print("Совпадение с сохранённым примером:", first_line == reference["short_completion"].strip())
assert first_line == reference["short_completion"].strip()
print("\nПолный ответ модели:\n" + result["text"])
write_json(ROOT / "samples/notebook_example.json", result)


Префикс: За зимой приходит
Короткое дополнение: весна
Совпадение с сохранённым примером: True

Полный ответ модели:
За зимой приходит весна
И я не знаю, как мне быть
Я хочу тебя обнять
Но ты не хочешь меня знать

Я хочу тебя обнять
Но ты не


## Свой префикс

Сначала выполните ячейку загрузки модели. В следующей ячейке замените `prompt` на любой текст: сверки с подборкой примеров здесь нет. Тройные кавычки позволяют вводить несколько строк. При `temperature=0` генерация жадная; при `temperature=0.85` используется случайная выборка.

In [7]:
prompt = """Сколько лет терпеть мне эту ночь"""

result = generate(
    model, tokenizer, prompt,
    seed=42,
    max_new_tokens=128,
    temperature=0.85,
)
print(result["text"])


Сколько лет терпеть мне эту ночь
Тобой управляет моя любовь
Сколько дней я прожил зря, вновь и вновь
И на сколько мне будет еще боль
Я найду свой путь и закрою глаза
Я найду ту дверь что была мне нужна
И в тот момент, когда я найду себя
И только тогда, когда мне будет пизда
